# Toy 2 — Label-Switching Mixture of Experts: A Provably Multimodal Posterior

This notebook is the Tier-1 "illustrative example" for the *Stochastic
Surrogate Constitutive Models* paper (`results.tex`, `sec:toy`): a
low-dimensional problem with a **known, provably multimodal** posterior, to
show where the unimodal Gaussian families fail and where the Langevin
generator does not.

## Why this posterior is exactly bimodal, not just empirically so

We build a 2-expert mixture of small neural networks,

```
y_hat(x) = pi_0(x) f_0(x) + pi_1(x) f_1(x)
```

where `f_0`, `f_1` are identical-architecture small NNs and the gate is a
**symmetric** construction: a scalar "pivot" network `g(x)` feeds *both*
gate logits as `[g(x), -g(x)]`, so `pi_0(x) = sigmoid(2 g(x))`,
`pi_1(x) = sigmoid(-2 g(x))`.

Consider the transformation that (a) swaps expert 0's and expert 1's
parameters, and (b) negates the pivot network's **final layer** only (weight
and bias; NOT its earlier layers -- see the symmetry-check cell below for why
that distinction matters whenever the activation isn't odd, e.g. `SiLU`).
This transformation leaves `y_hat(x)` **exactly** unchanged at every `x` (we
verify this numerically below, not just assert it), and -- because expert 0
and expert 1 share one prior and the pivot's final-layer prior is zero-mean
and symmetric -- it also leaves the joint prior density unchanged. Combined,
the true posterior is **exactly** invariant under this transformation: it has
(at least) two mirror-image modes, exchanged by swap+negate, with the only
possible fixed point at the measure-zero set where the pivot's final layer is
identically zero.

This is the same mechanism as label-switching in any exchangeable mixture
model; what is unusual here is that the exchangeability is engineered to be
*exact and provable*, by construction, in a NN-based mixture, rather than
asserted.

We then train all 5 families and show: `Deterministic`/`IIDGaussian` (mean-field)/
`Gaussian` (full-covariance)/`LowRankGaussian` each commit to **one** mode per
random seed (population-level bimodality only shows up *across* seeds), while
a single `LangevinGroup` run's particle cloud can cover **both** modes at
once.

In [ ]:
import time

_notebook_start = time.time()

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from pypolymix.parameter_groups import DeterministicGroup, GaussianGroup, IIDGaussianGroup, LangevinGroup, LowRankGaussianGroup
from pypolymix.parameter_groups.kl_estimators import CrossEntropyOnly, DiagonalGaussianEntropyKL, KNNEntropyKL
from pypolymix.priors import IIDGaussianPrior
from pypolymix.surrogate_models import MixtureOfExperts, NeuralNetwork
from pypolymix.surrogate_models.base import SurrogateModel
from pypolymix import StochasticModel

torch.set_default_dtype(torch.float64)

PAPER_DAT_DIR = pathlib.Path.home() / "projects/salt/doc/paper/stochastic_surrogates/dat"

FAMILIES_UNIMODAL = ["deterministic", "iid", "full", "lowrank"]
FAMILY_LABELS = {
    "deterministic": "Deterministic (MAP)",
    "iid": "Mean-field Gaussian",
    "full": "Full-covariance Gaussian",
    "lowrank": "Low-rank Gaussian",
    "langevin": "Langevin",
}
FAMILY_COLORS = {
    "deterministic": "tab:gray",
    "iid": "tab:blue",
    "full": "tab:green",
    "lowrank": "tab:orange",
    "langevin": "tab:red",
}

## Data

The same `y(x) = sin(pi x) + sign(x) + noise` 2-regime problem as
`moe.ipynb`, which already motivates needing more than one expert: a single
smooth model cannot capture the discontinuity at `x = 0`.

In [ ]:
torch.manual_seed(0)
n_data = 50
x_data = 2 * torch.rand(n_data, 1) - 1
y_data = torch.sin(torch.pi * x_data) + torch.sign(x_data) + 0.05 * torch.randn(n_data, 1)

_, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(x_data, y_data)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()

## The symmetric architecture

`SymmetricGate` wraps a scalar "pivot" `SurrogateModel` (any architecture) and
returns the 2-way softmax of `[g(x), -g(x)]`. This mirrors `GatingNetwork`'s
interface exactly (`forward` returns already-softmaxed weights), so it drops
straight into `MixtureOfExperts`.

In [ ]:
class SymmetricGate(SurrogateModel):
    '''2-expert gate with logits [g(x), -g(x)] for a scalar pivot network g.

    Swapping which expert is "expert 0" vs "expert 1" and negating g's
    FINAL layer only leaves the mixture weights -- and hence the whole MoE
    output -- exactly unchanged (verified numerically below).
    '''

    def __init__(self, pivot_net: SurrogateModel):
        super().__init__()
        self.pivot_net = pivot_net
        self.num_inputs = pivot_net.num_inputs
        self.num_experts = 2

    def forward(self, x: torch.Tensor, params: torch.Tensor) -> torch.Tensor:
        g = self.pivot_net(x, params)  # (S, B, 1)
        logits = torch.cat([g, -g], dim=-1)  # (S, B, 2)
        return F.softmax(logits, dim=-1)

    def num_params(self) -> int:
        return self.pivot_net.num_params()


EXPERT_WIDTH, EXPERT_DEPTH = 4, 2
GATE_WIDTH, GATE_DEPTH = 4, 1


def build_moe():
    expert0 = NeuralNetwork(num_inputs=1, num_outputs=1, width=EXPERT_WIDTH, depth=EXPERT_DEPTH, activation=F.silu)
    expert1 = NeuralNetwork(num_inputs=1, num_outputs=1, width=EXPERT_WIDTH, depth=EXPERT_DEPTH, activation=F.silu)
    pivot = NeuralNetwork(num_inputs=1, num_outputs=1, width=GATE_WIDTH, depth=GATE_DEPTH, activation=F.silu)
    gate = SymmetricGate(pivot)
    return MixtureOfExperts(experts=[expert0, expert1], gating_network=gate)


moe = build_moe()
N_EXPERT = moe.experts[0].num_params()
N_GATE = moe.gating_network.num_params()
N_TOTAL = moe.num_params()
print(f"expert params: {N_EXPERT} each, gate (pivot) params: {N_GATE}, total: {N_TOTAL}")

### Numerical proof of the swap symmetry

This is not an assertion -- we construct a random parameter vector, apply the
swap+negate transformation, and check the MoE's output is identical (up to
floating point) at every test point.

In [ ]:
# Pivot layout for width=GATE_WIDTH, depth=GATE_DEPTH=1: [W0 (GATE_WIDTH), b0 (GATE_WIDTH), W1 (GATE_WIDTH), b1 (1)]
n_gate_hidden = GATE_WIDTH * 1 + GATE_WIDTH  # W0 + b0
n_gate_final = GATE_WIDTH * 1 + 1  # W1 + b1
assert n_gate_hidden + n_gate_final == N_GATE

torch.manual_seed(1)
theta_e0 = torch.randn(1, N_EXPERT)
theta_e1 = torch.randn(1, N_EXPERT)
theta_g = torch.randn(1, N_GATE)
theta_g_hidden, theta_g_final = theta_g[:, :n_gate_hidden], theta_g[:, n_gate_hidden:]

theta = torch.cat([theta_e0, theta_e1, theta_g], dim=1)
theta_swapped = torch.cat([theta_e1, theta_e0, theta_g_hidden, -theta_g_final], dim=1)

x_probe_grid = torch.linspace(-1, 1, 51).unsqueeze(-1)
with torch.no_grad():
    y = moe(x_probe_grid, theta)
    y_swapped = moe(x_probe_grid, theta_swapped)

max_abs_diff = (y - y_swapped).abs().max().item()
print(f"max |y(theta) - y(swap(theta))| over the test grid: {max_abs_diff:.2e}")
assert max_abs_diff < 1e-10, "swap symmetry check FAILED -- see markdown above for the required transform"
print("Swap symmetry check PASSED: the two configurations are exactly observationally equivalent.")
print("(Negating the pivot's hidden layer too -- not just its final layer -- would break this,")
print(" since SiLU is not an odd activation; only the final linear layer's sign flip is exact.)")

## Training all 5 families

We use a single **isotropic, zero-mean** Gaussian prior
`IIDGaussianPrior(mean=0, std=sigma_prior)` applied uniformly across the
*entire* pooled parameter vector (both experts and the gate pivot). This is
the simplest prior that is simultaneously invariant under (a) swapping the
two expert blocks and (b) negating any subset of coordinates (in particular
the pivot's final layer) -- exactly the symmetry the architecture needs, with
no special-casing of blocks required.

Per `methodology.tex`'s `sec:partial`, the mean-field family would normally
use independent per-layer groups while full-covariance/low-rank/Langevin pool
everything into one group (to capture cross-layer correlations). For this
toy we pool *all* families into one group of the appropriate type for
simplicity; mean-field's `IIDGaussianGroup` is already elementwise
independent regardless of how the pooling is organized, so nothing about the
comparison changes.

Unlike Toy 1, we deliberately do **not** center the prior at a deterministic
point estimate here: a MAP fit would land on an arbitrary one of the two
modes, and centering the prior there would silently and unfairly break the
exchangeability the whole demonstration depends on. This is an intentional
departure from Toy 1's (and the real experiments') whitening-around-a-point-
estimate recipe, not an oversight.

In [ ]:
SIGMA_PRIOR = 1.5
SIGMA_OBS = 0.1


def make_group(kind, *, langevin_kwargs=None):
    prior = IIDGaussianPrior(mean=0.0, std=SIGMA_PRIOR)
    if kind == "deterministic":
        return DeterministicGroup("pooled", N_TOTAL)
    if kind == "iid":
        return IIDGaussianGroup("pooled", N_TOTAL, prior=prior)
    if kind == "full":
        return GaussianGroup("pooled", N_TOTAL, prior=prior)
    if kind == "lowrank":
        return LowRankGaussianGroup("pooled", N_TOTAL, rank=8, prior=prior)
    if kind == "langevin":
        kwargs = dict(
            num_particles=128, num_diffusion_steps=40, step_size=5e-3, init_std=3.0,
            kl_estimator=DiagonalGaussianEntropyKL(),
        )
        if langevin_kwargs:
            kwargs.update(langevin_kwargs)
        score_model = NeuralNetwork(num_inputs=N_TOTAL, num_outputs=N_TOTAL, width=64, depth=2)
        return LangevinGroup("pooled", N_TOTAL, score_model=score_model, prior=prior, **kwargs)
    raise ValueError(kind)


def train(seed, kind, num_epochs=4000, num_samples=32, lr=1e-2, langevin_kwargs=None):
    torch.manual_seed(seed)
    moe_local = build_moe()
    group = make_group(kind, langevin_kwargs=langevin_kwargs)
    model = StochasticModel(moe_local, group)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=5 * lr, total_steps=num_epochs)
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        params = model.sample_parameters(num_samples=num_samples)
        pred = moe_local(x_data, params)
        target = y_data.unsqueeze(0).expand_as(pred)
        nll_sum = 0.5 * ((pred - target) / SIGMA_OBS).pow(2).sum(dim=(1, 2)).mean()
        kl = model.distribution_loss()
        loss = nll_sum + kl
        loss.backward()
        optimizer.step()
        scheduler.step()
    return group, model, moe_local


X_PROBE = torch.tensor([[0.5]])  # swap-sensitive probe point: pi_0(0.5) vs pi_1(0.5) distinguishes the two modes


def probe_values(group, moe_local, kind, num_samples=2000):
    # g(x=0.5) under q (or the single point estimate for "deterministic")
    gate = moe_local.gating_network
    with torch.no_grad():
        if kind == "deterministic":
            params = group.params.unsqueeze(0)
        else:
            params = group.sample_parameters(num_samples)
        gate_params = params[:, -gate.num_params():]
        g = gate.pivot_net(X_PROBE, gate_params)
    return g.squeeze(-1).squeeze(-1)

### Multi-seed runs for the 4 unimodal families

Each seed trains an independent model from scratch. We record the pivot
probe `g(x=0.5)` for each: for `deterministic` this is a single point; for
the 3 Gaussian families we additionally record the *within-seed* spread
(each seed's own posterior should be tightly unimodal -- confident about
*which* mode it found, even though different seeds can disagree on *which*
mode that is).

In [ ]:
N_SEEDS = 10
seed_results = {kind: [] for kind in FAMILIES_UNIMODAL}

for kind in FAMILIES_UNIMODAL:
    t0 = time.time()
    for seed in range(N_SEEDS):
        group, model, moe_local = train(seed, kind)
        probe = probe_values(group, moe_local, kind)
        seed_results[kind].append(probe.numpy())
    print(f"{FAMILY_LABELS[kind]:28s} {N_SEEDS} seeds trained in {time.time() - t0:.1f}s")

### Single Langevin run

One `LangevinGroup` fit, trained once. If the particle cloud covers both
modes, this single run should show roughly balanced positive/negative
`g(x=0.5)` mass, unlike any single seed of the families above.

In [ ]:
t0 = time.time()
langevin_group, langevin_model, langevin_moe = train(0, "langevin")
langevin_probe = probe_values(langevin_group, langevin_moe, "langevin", num_samples=4000).numpy()
print(f"Langevin trained in {time.time() - t0:.1f}s")
print(f"fraction of particles with g(x=0.5) > 0: {(langevin_probe > 0).mean():.3f}")
print(f"g(x=0.5) 5/50/95 percentiles: {np.percentile(langevin_probe, [5, 50, 95])}")

## The headline figure: collapse (across seeds) vs. coverage (within one run)

In [ ]:
fig, axes = plt.subplots(1, len(FAMILIES_UNIMODAL) + 1, figsize=(3.2 * (len(FAMILIES_UNIMODAL) + 1), 3.2), sharey=True)

for ax, kind in zip(axes[:-1], FAMILIES_UNIMODAL):
    seed_means = [p.mean() if kind != "deterministic" else p.item() for p in seed_results[kind]]
    ax.scatter(np.zeros(N_SEEDS), seed_means, color=FAMILY_COLORS[kind], s=40)
    ax.axhline(0, color="black", lw=0.5, ls=":")
    n_pos = sum(1 for m in seed_means if m > 0)
    ax.set_title(f"{FAMILY_LABELS[kind]}\n{n_pos}/{N_SEEDS} seeds > 0", fontsize=9)
    ax.set_xticks([])

ax = axes[-1]
ax.hist(langevin_probe, bins=30, color=FAMILY_COLORS["langevin"], orientation="horizontal")
ax.axhline(0, color="black", lw=0.5, ls=":")
frac_pos = (langevin_probe > 0).mean()
ax.set_title(f"Langevin (1 run)\n{frac_pos:.0%} of particles > 0", fontsize=9)

axes[0].set_ylabel("pivot probe g(x=0.5)")
plt.tight_layout()

**Expected finding (to confirm once this notebook is run end to end via
papermill):** each seed of the 4 unimodal families should land confidently on
*one* sign of `g(x=0.5)` or the other (collapse within a run), with the sign
varying *across* seeds (revealing the population-level bimodality one seed
alone cannot see) -- while the single Langevin run's particle histogram
should show mass on both sides of zero (coverage within one run). Getting
Langevin to actually achieve this took one deliberate hyperparameter choice
worth stating plainly: `init_std` must be wide enough that particles are
initially spread across *both* basins before the score network sharpens
around whichever mode random early-training noise favors (`init_std=3.0`
here; a narrower, default-ish `init_std=1.0` was tried first and collapsed to
a single mode in early development of this notebook -- an informative
negative result in its own right, analogous to
`kl_estimators_comparison.ipynb`'s own Part 3 finding that naive Langevin
settings can fail to discover a known second mode).

## Prior-parameterization and KL-estimator sensitivity (Langevin)

The second half of the requested KL-representation exploration: does
Langevin's ability to cover both modes depend sensitively on the prior scale
or on the choice of `kl_estimator`? We sweep `sigma_prior` and the estimator
jointly and report the resulting positive-probe fraction (0.5 = perfectly
balanced coverage; 0 or 1 = collapsed to one mode).

In [ ]:
PRIOR_STDS = [0.75, 1.5, 3.0]
ESTIMATORS = {
    "CrossEntropyOnly": CrossEntropyOnly(),
    "DiagonalGaussianEntropyKL": DiagonalGaussianEntropyKL(),
    "KNNEntropyKL": KNNEntropyKL(),
}

sensitivity_rows = []
t0 = time.time()
for sigma_prior_sweep in PRIOR_STDS:
    for est_name, estimator in ESTIMATORS.items():
        torch.manual_seed(0)
        moe_local = build_moe()
        prior = IIDGaussianPrior(mean=0.0, std=sigma_prior_sweep)
        score_model = NeuralNetwork(num_inputs=N_TOTAL, num_outputs=N_TOTAL, width=64, depth=2)
        group = LangevinGroup(
            "pooled", N_TOTAL, score_model=score_model, prior=prior,
            num_particles=128, num_diffusion_steps=40, step_size=5e-3, init_std=3.0,
            kl_estimator=estimator,
        )
        model = StochasticModel(moe_local, group)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=5e-2, total_steps=4000)
        for epoch in range(4000):
            optimizer.zero_grad()
            params = model.sample_parameters(num_samples=32)
            pred = moe_local(x_data, params)
            target = y_data.unsqueeze(0).expand_as(pred)
            nll_sum = 0.5 * ((pred - target) / SIGMA_OBS).pow(2).sum(dim=(1, 2)).mean()
            kl = model.distribution_loss()
            loss = nll_sum + kl
            loss.backward()
            optimizer.step()
            scheduler.step()
        probe = probe_values(group, moe_local, "langevin", num_samples=2000).numpy()
        frac_pos = (probe > 0).mean()
        sensitivity_rows.append((sigma_prior_sweep, est_name, frac_pos, probe.std()))
        print(f"sigma_prior={sigma_prior_sweep:.2f} estimator={est_name:28s} frac_pos={frac_pos:.3f} std={probe.std():.3f}")
print(f"sensitivity sweep total time: {time.time() - t0:.1f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for est_name in ESTIMATORS:
    rows = [r for r in sensitivity_rows if r[1] == est_name]
    ax.plot([r[0] for r in rows], [r[2] for r in rows], "o-", label=est_name)
ax.axhline(0.5, color="black", ls=":", lw=1, label="perfectly balanced")
ax.set_xlabel("prior std (sigma_prior)")
ax.set_ylabel("fraction of particles with g(x=0.5) > 0")
ax.set_ylim(-0.05, 1.05)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()

**Expected finding (to confirm once run):** if balanced coverage
(`frac_pos` near 0.5) holds across most/all prior-std and estimator
combinations, that is evidence the two-mode coverage demonstrated above is a
robust property of the Langevin mechanism itself, not a fragile artifact of
one specific hyperparameter choice. If some combinations collapse (`frac_pos`
near 0 or 1) while others don't, that is itself the reportable finding --
following `kl_estimators_comparison.ipynb`'s own "report observed, not
hypothesized" convention, a collapse here would narrow the claim rather than
invalidate it (Langevin *can* cover both modes under suitable settings,
rather than *always* does regardless of configuration).

## Exporting data tables for the paper's tikz/pgfplots pipeline

In [ ]:
PAPER_DAT_DIR.mkdir(parents=True, exist_ok=True)

# Seed-probe values for the unimodal families (one row per seed)
with open(PAPER_DAT_DIR / "toy_moe_seed_probes.txt", "w") as f:
    f.write("family seed probe_mean\n")
    for kind in FAMILIES_UNIMODAL:
        for seed, p in enumerate(seed_results[kind]):
            val = p.item() if kind == "deterministic" else float(np.mean(p))
            f.write(f"{kind} {seed} {val:.6f}\n")

# Langevin particle-cloud probe values (one row per particle)
np.savetxt(
    PAPER_DAT_DIR / "toy_moe_langevin_particle_probes.txt",
    langevin_probe.reshape(-1, 1),
    header="probe_value",
    comments="",
)

# Prior/estimator sensitivity sweep
with open(PAPER_DAT_DIR / "toy_moe_kl_sensitivity.txt", "w") as f:
    f.write("sigma_prior estimator frac_pos probe_std\n")
    for sigma_prior_sweep, est_name, frac_pos, probe_std in sensitivity_rows:
        f.write(f"{sigma_prior_sweep} {est_name} {frac_pos:.6f} {probe_std:.6f}\n")

print("wrote dat/ tables to", PAPER_DAT_DIR)

In [ ]:
elapsed = time.time() - _notebook_start
print(f"Total notebook wall-clock time: {elapsed:.1f} s ({elapsed / 60:.2f} min)")